## Defining architechure and train function


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/train/', transform=transform_train)
test_dataset  = datasets.ImageFolder('content/fer2013plus/fer2013plus/fer2013/test/', transform=transform_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)
test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super(CBAM, self).__init__()
        # Channel attention: squeeze global info then excite important channels
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.channel_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.channel_sigmoid = nn.Sigmoid()
        # Spatial attention: highlight important locations using pooled feature maps
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.spatial_sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Channel attention
        avg_out = self.channel_fc(self.avg_pool(x))
        max_out = self.channel_fc(self.max_pool(x))
        ca = self.channel_sigmoid((avg_out + max_out).view(x.size(0), x.size(1), 1, 1))
        x = x * ca
        # Spatial attention
        avg_s = x.mean(dim=1, keepdim=True)
        max_s, _ = x.max(dim=1, keepdim=True)
        sa = self.spatial_sigmoid(self.spatial_conv(torch.cat([avg_s, max_s], dim=1)))
        return x * sa
# ────────────────────────────────────────────────────────────────────────────────

class FER_DCNN(nn.Module):
    def __init__(self):
        super(FER_DCNN, self).__init__()

        # Helper to create a block with 2 convs + CBAM attention per block
        def conv_block(in_f, out_f, dropout_rate):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                CBAM(out_f),          # <-- CHANGE 1: Attention after each conv block
                nn.MaxPool2d(2),
                nn.Dropout(dropout_rate)
            )

        # 1. Conv block count: 3 sets
        # 2. Filter counts: 64, 128, 256
        # 3. Dropout (conv): Rate 0.3 for all blocks
        self.features = nn.Sequential(
            conv_block(1, 64, 0.3),   # Block 1
            conv_block(64, 128, 0.3),  # Block 2
            conv_block(128, 256, 0.3)  # Block 3
        )

        # 4. Dense layers: 1 dense, 128 neurons
        # Spatial size logic: 48 -> 24 -> 12 -> 6 (after 3 MaxPools)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),          # 5. Dropout (dense): Rate 0.4
            nn.Linear(128, 8)         # Keeping 8 output neurons as per your code
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FER_DCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scaler = GradScaler("cuda", enabled=torch.cuda.is_available())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast("cuda", enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach()

    return (total_loss / len(loader)).item()


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.detach()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    return (total_loss / len(loader)).item(), acc

## Training Model

In [4]:
# Training loop
best_acc, best_val_loss = 0, float('inf')
patience, no_improve_count, epoch = 10, 0, 0

while True:
    train_loss = train_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
)
    val_loss, val_acc = evaluate(
    model,
    test_loader,
    criterion,
    device
)

    scheduler.step(val_loss)   # Fix: step on val_loss, not train_loss

    if val_acc > best_acc + 1e-4:
        best_acc = val_acc
        no_improve_count = 0
        torch.save(model.state_dict(), "best_fer_model.pth")
    else:
        no_improve_count += 1

    print(f"Epoch {epoch+1}: TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"ValAcc={val_acc:.4f}  NoImprove={no_improve_count}/{patience}")

    if no_improve_count >= patience:
        print(f"Converged at epoch {epoch+1}. Best Val Acc: {best_acc:.4f}")
        break

    epoch += 1

Epoch 1: TrainLoss=1.4703  ValLoss=1.1622  ValAcc=61.0368  NoImprove=0/10
Epoch 2: TrainLoss=1.0184  ValLoss=0.9341  ValAcc=66.0516  NoImprove=0/10
Epoch 3: TrainLoss=0.8694  ValLoss=0.8904  ValAcc=68.3054  NoImprove=0/10
Epoch 4: TrainLoss=0.7952  ValLoss=0.7883  ValAcc=71.8975  NoImprove=0/10
Epoch 5: TrainLoss=0.7284  ValLoss=0.7100  ValAcc=74.5175  NoImprove=0/10
Epoch 6: TrainLoss=0.6899  ValLoss=0.6917  ValAcc=75.1092  NoImprove=0/10
Epoch 7: TrainLoss=0.6509  ValLoss=0.7065  ValAcc=75.0669  NoImprove=1/10
Epoch 8: TrainLoss=0.6169  ValLoss=0.6553  ValAcc=76.3488  NoImprove=0/10
Epoch 9: TrainLoss=0.5881  ValLoss=0.6216  ValAcc=77.9828  NoImprove=0/10
Epoch 10: TrainLoss=0.5657  ValLoss=0.6863  ValAcc=75.8417  NoImprove=1/10
Epoch 11: TrainLoss=0.5463  ValLoss=0.6399  ValAcc=77.5743  NoImprove=2/10
Epoch 12: TrainLoss=0.5261  ValLoss=0.6231  ValAcc=78.3913  NoImprove=0/10
Epoch 13: TrainLoss=0.4980  ValLoss=0.7774  ValAcc=74.6161  NoImprove=1/10
Epoch 14: TrainLoss=0.4440  ValLos